# Solutions · 01 · The first law and energy balances

Worked solutions to the exercises of [notebook 01](../notebooks/01_first_law_and_energy_balances.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engthermo import idealgas
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A gas turbine's exhaust (mostly air, 1.5 kg/s at 750 K) heats water from 20 to 90 degC in a heat-recovery exchanger while cooling to 400 K. How much water can be heated per second? (Liquid water: cp = 4.18 kJ/(kg K).)

In [2]:
m_gas = 1.5
air = idealgas.Mixture({"N2": 0.79, "O2": 0.21})
Q = m_gas / air.M * air.enthalpy_change(400.0, 750.0)                 # J/s given up by the exhaust
m_water = Q / (4180 * 70)
print(f"heat recovered {Q/1e3:.1f} kW -> {m_water:.2f} kg/s of water heated from 20 to 90 degC")

heat recovered 553.5 kW -> 1.89 kg/s of water heated from 20 to 90 degC


The exhaust's enthalpy change between 750 and 400 K (mean heat capacity about 30 J/(mol K)) heats nearly two
kilograms of water per second by 70 K. Heat-recovery steam generators and district-heating exchangers on gas
turbines run exactly this balance.

## Exercise 2

Three-stage compression of air from 1 to 27 bar with intercooling to 300 K: choose the intermediate pressures, compute the work, and compare with single-stage compression and with isothermal compression (RT ln(p2/p1)).

In [3]:
T1, p1, p2 = 300.0, 1e5, 27e5
r = (p2 / p1) ** (1 / 3)                                              # equal pressure ratios per stage
stages = [(p1, p1 * r), (p1 * r, p1 * r**2), (p1 * r**2, p2)]
w3 = sum(idealgas.enthalpy_change("air", T1, idealgas.isentropic_T("air", T1, a, b)) for a, b in stages)
w1 = idealgas.enthalpy_change("air", T1, idealgas.isentropic_T("air", T1, p1, p2))
w_iso = 8.314462618 * T1 * np.log(p2 / p1)
print(f"intermediate pressures {p1*r/1e5:.2f} and {p1*r**2/1e5:.2f} bar")
print(f"work: single stage {w1/1e3:.2f}, three stages {w3/1e3:.2f}, isothermal {w_iso/1e3:.2f} kJ/mol")

intermediate pressures 3.00 and 9.00 bar
work: single stage 13.54, three stages 9.65, isothermal 8.22 kJ/mol


Three stages with intercooling save nearly 30 % of the single-stage work and come within about 17 % of the
isothermal minimum. Each intercooler also lowers the outlet temperature, which matters for the mechanical
design of the compressor and for compressed-air storage.

## Exercise 3

**Implement it yourself:** write `isentropic_T(gas, T1, p1, p2)` yourself using `entropy_change` and `brentq`, and reproduce the compressor outlet temperature of the notebook.

In [4]:
def isentropic_T(gas, T1, p1, p2):
    return optimize.brentq(lambda T: idealgas.entropy_change(gas, T1, T, p1, p2), T1 if p2 > p1 else 150.0, 4000.0 if p2 > p1 else T1)
print(f"by hand: {isentropic_T('air', 300.0, 1e5, 1e6):.2f} K;  library: {idealgas.isentropic_T('air', 300.0, 1e5, 1e6):.2f} K")

by hand: 574.17 K;  library: 574.17 K


The isentropic condition is one equation in one unknown - the entropy change from the inlet state must be
zero - and a bracketed root finder solves it for any heat-capacity function.